## Week 1 Challenge — Brochure Generator with Telugu to English Translation

## This notebook extends Ed Donner's Day 5 brochure generator with two key additions:

### What this does
- Scrapes the Telugu news website and generates a company brochure in Telugu
- Automatically translates the Telugu brochure into English using a second LLM call
- Demonstrates chaining multiple LLM calls together for a complete pipeline

### How it works
1. BeautifulSoup scrapes all links from the website
2. First LLM call → filters relevant links and generates Telugu brochure
3. Second LLM call → translates Telugu brochure to English
4. Both outputs displayed as clean Markdown

### Key concepts demonstrated
- Multiple LLM calls chained together
- Prompt engineering for translation
- Web scraping with BeautifulSoup
- Error handling for bad URLs

### Requirements
- OpenAI API key in `.env` file
- Install dependencies: `uv pip install beautifulsoup4 requests openai`

---
*Built as a Week 1 Challenge extension for the Udemy course: AI Engineer Core Track — LLM Engineering, RAG, QLoRA, Agents by Ed Donner.*

In [ ]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [ ]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

In [ ]:
links = fetch_website_links("https://www.eenadu.net/")
links

## First step: Have GPT-5-nano figure out which links are relevant


In [ ]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the news company,
such as links to an About page, Editorial Team page, News Categories/Sections, 
Contact page, or Advertising/Reach pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "editorial team page", "url": "https://another.full.url/team"}
    ]
}
"""

In [ ]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
print(get_links_user_prompt("https://www.eenadu.net/"))

In [ ]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [ ]:
select_relevant_links("https://www.eenadu.net/")

In [ ]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://www.eenadu.net/")

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [ ]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        try:
            result += f"\n\n### Link: {link['type']}\n"
            result += fetch_website_contents(link["url"])
        except Exception as e:
            print(f"Skipping {link['url']}: {e}")
            continue
    return result

In [ ]:
print(fetch_page_and_all_relevant_links("https://www.eenadu.net/"))

In [ ]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages 
from a Telugu news company website and creates a short brochure about the 
company in Telugu language for prospective customers, investors and recruits.
Respond in markdown without code blocks.
"""

In [ ]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a Telugu news company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in Telugu language in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000]
    return user_prompt

In [ ]:
get_brochure_user_prompt("Eenadu", "https://www.eenadu.net/")

In [ ]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    telugu_brochure = response.choices[0].message.content
    display(Markdown(telugu_brochure))

In [ ]:
create_brochure("Eenadu", "https://eenadu.net/")

###  Creating Translation Prompts?

Instead of asking the model to generate the brochure directly in English, we use two separate LLM calls:

1. **First LLM call** — generates the brochure in Telugu using the website content
2. **Second LLM call** — translates the Telugu brochure to English

Each prompt does ONE thing only:
- `brochure_system_prompt` → focuses purely on generating a good brochure
- `translation_system_prompt` → focuses purely on accurate translation

This makes the code modular and reusable — the translation function can be used to translate any content, not just brochures!

In [ ]:
translation_system_prompt = """
You are an expert translator fluent in Telugu and English.
You will be given a company brochure written in Telugu.
Translate the entire brochure accurately and naturally into English.
Maintain the same formatting, structure and tone as the original.
Respond in markdown without code blocks.
"""

In [ ]:
def get_translation_user_prompt(telugu_brochure):
    user_prompt = f"""Here is a company brochure written in Telugu.
Translate it accurately and naturally into English.

{telugu_brochure}
"""
    return user_prompt

In [ ]:
def create_brochure(company_name, url):
    # First API call → generates Telugu brochure with streaming
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
        stream=True
    )
    telugu_brochure = ""
    for chunk in response:
        telugu_brochure += chunk.choices[0].delta.content or ""
    display(Markdown(telugu_brochure))
    
    # Second API call → translates Telugu brochure to English with streaming
    translation_response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": translation_system_prompt},
            {"role": "user", "content": get_translation_user_prompt(telugu_brochure)}
        ],
        stream=True
    )
    english_brochure = ""
    for chunk in translation_response:
        english_brochure += chunk.choices[0].delta.content or ""
    display(Markdown(english_brochure))

In [ ]:
create_brochure("Eenadu", "https://eenadu.net/")